# ML-1M SASRec FullCE — canonical training

Dedicated training notebook for the exact SASRec + FullCE baseline used in the SparseWalker ML-1M comparison. It runs only one model and writes the full training history to Drive.

Config: d=64, 2 blocks, 1 head, dropout=0.1, max_len=200, FullCE, Adam lr=1e-3, batch=128, leave-two-out split, full-catalog evaluation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, subprocess, shutil
REPO='/content/Sparsewalker'
BRANCH='agent/sasrec-ml1m-training'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO],check=True)

import torch
print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('BRANCH',BRANCH)


In [ ]:
OUT='/content/drive/MyDrive/sparsewalker_sasrec_ml1m'
SCRIPT=f'{REPO}/benchmarks/run_esasrec_2x2.py'
cmd=[sys.executable,'-u',SCRIPT,
     '--dataset','ml1m',
     '--seed','42',
     '--models','SASRec+FullCE',
     '--batch-size','128',
     '--eval-batch-size','1024',
     '--max-epochs','100',
     '--patience','50',
     '--eval-every','5',
     '--lr','1e-3',
     '--weight-decay','0',
     '--output-dir',OUT]
print('RUNNING',' '.join(cmd),flush=True)
subprocess.run(cmd,cwd=REPO,check=True)


## Inspect loss curve and epoch 10

In [ ]:
import pandas as pd
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_sasrec_ml1m/ml1m/seed42/SASRec_FullCE/history.csv')
df=pd.read_csv(p)
display(df[['epoch','loss','NDCG@10','HR@10','MRR@10']])
print('EPOCH 10')
display(df[df['epoch']==10][['epoch','loss','NDCG@10','HR@10','MRR@10']])
